# S1 — Loading and understanding BodyM

Run this in **Google Colab** (Runtime → Run all) or Jupyter on your laptop.
Each step prints what it finds, so you see the real file layout rather than trusting assumptions.

What BodyM gives you per person:
- **inputs**: height, weight, gender, a front silhouette and a side silhouette
- **outputs (answers)**: 14 body measurements in cm, computed on a 3D scan

Licence: CC BY-NC 4.0 — fine for school, not for commercial use.

## Step 1 — Install the tools

In [ ]:
!pip -q install awscli pandas pillow matplotlib scikit-learn

## Steps 2–3 — Get the data (Drive if saved, else download once)
See docs/DATA.md and DISCUSSION D-010: one `.tar` on Drive is far faster than thousands of small files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"

if os.path.exists(DATA_DIR):
    print("Already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    print("Restoring from Drive...")
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    print("First time: downloading from Amazon, then saving to Drive...")
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym
!du -sh {DATA_DIR}

## Step 4 — Map the folder structure
Counts files per folder. Expect one folder per split (training, Test-A, Test-B),
each with CSV tables and folders of silhouette images.

In [ ]:
import collections
from pathlib import Path

root = Path(DATA_DIR)
counts = collections.Counter()
exts = collections.defaultdict(collections.Counter)
for p in root.rglob("*"):
    if p.is_file():
        counts[str(p.parent.relative_to(root))] += 1
        exts[str(p.parent.relative_to(root))][p.suffix.lower()] += 1

for folder in sorted(counts):
    print(f"{folder:45s} {counts[folder]:6d} files  {dict(exts[folder])}")

## Step 5 — Read every table
For each CSV: its size, its columns and the first rows.
This is where you learn the column names (subject ID, height, weight, the 14 measurements, photo IDs).

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)

csvs = sorted(root.rglob("*.csv"))
tables = {}
for f in csvs:
    df = pd.read_csv(f)
    key = str(f.relative_to(root))
    tables[key] = df
    print("=" * 90)
    print(key, df.shape)
    print("columns:", list(df.columns))
    print(df.head(3).to_string())

## Step 6 — Tell the code which columns are which
Read Step 5's output, then set these names. The defaults are guesses; the cell checks them and
tells you what to change if they're wrong.

In [ ]:
# Which top-level folders are the splits (from Step 4):
SPLITS = {"train": None, "testA": None, "testB": None}
for d in sorted(p.name for p in root.iterdir() if p.is_dir()):
    low = d.lower()
    if "train" in low: SPLITS["train"] = d
    elif low.endswith("a") or "test_a" in low or "testa" in low: SPLITS["testA"] = d
    elif low.endswith("b") or "test_b" in low or "testb" in low: SPLITS["testB"] = d
print("splits found:", SPLITS)

def find_col(df, *keywords):
    for c in df.columns:
        if all(k in c.lower() for k in keywords):
            return c
    return None

def split_tables(split_dir):
    return {k: v for k, v in tables.items() if k.startswith(split_dir + "/")}

for name, d in SPLITS.items():
    if d is None: continue
    print(f"\n[{name}] tables:")
    for k, df in split_tables(d).items():
        print("  ", k, "| id-like columns:", [c for c in df.columns if "id" in c.lower()])

## Step 7 — Join the tables
Tables share `subject_id`. The result has **one row per photo pair**, not per person: a person with 19 pairs appears 19 times. Every score must use one row per person (DISCUSSION D-007).

In [ ]:
from functools import reduce

def build_split(split_dir):
    dfs = list(split_tables(split_dir).values())
    def merge(a, b):
        common = [c for c in a.columns if c in b.columns and "id" in c.lower()]
        if not common:
            common = [c for c in a.columns if c in b.columns]
        return a.merge(b, on=common, how="inner") if common else a
    return reduce(merge, dfs)

data = {name: build_split(d) for name, d in SPLITS.items() if d}
for name, df in data.items():
    print(name, df.shape)
    print(list(df.columns), "\n")

## Step 8 — Look at one person
Shows the two silhouettes next to their numbers. This is exactly what the model sees (inputs)
and must predict (outputs).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

train_dir = root / SPLITS["train"]
img_folders = sorted({p.parent for p in train_dir.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg")})
print("image folders:", [str(f.relative_to(root)) for f in img_folders])

df = data["train"]
photo_col = find_col(df, "photo") or find_col(df, "image")
row = df.iloc[0]
print("\nperson:\n", row.to_string())

if photo_col:
    pid = str(row[photo_col])
    hits = [p for f in img_folders for p in f.glob(pid + ".*")]
    fig, axes = plt.subplots(1, len(hits), figsize=(4 * len(hits), 6))
    axes = [axes] if len(hits) == 1 else axes
    for ax, p in zip(axes, hits, strict=False):
        im = Image.open(p)
        ax.imshow(im, cmap="gray"); ax.set_title(f"{p.parent.name}\n{im.size}"); ax.axis("off")
    plt.show()
else:
    print("No photo-ID column found: check Step 5 and set photo_col by hand.")

## Step 9 — Sanity-check the numbers
Ranges should look human, in cm (e.g. waist 60–130). Also checks for missing values.

In [ ]:
num = data["train"].select_dtypes("number")
print(num.describe().T[["count", "mean", "min", "max"]].round(1).to_string())
print("\nmissing values per column:\n", data["train"].isna().sum()[lambda s: s > 0])

## Step 10 — First baseline: height + weight + sex, scored per person
Trained on one row per training person, scored on one row per Test-A person. The printed counts (2018 and 87) confirm it. Any photo model must beat this table.

In [ ]:
from sklearn.linear_model import LinearRegression

# One row per PERSON (D-007). Doing this after selecting the tables matters: an earlier version
# de-duplicated first and was then overwritten by a later reassignment (docs/CHALLENGES.md).
tr = data["train"].drop_duplicates("subject_id")
te = data["testA"].drop_duplicates("subject_id")
print("train people:", len(tr), "| testA people:", len(te))

h_col, w_col, g_col = "height_cm", "weight_kg", "gender"
targets = ["ankle", "arm-length", "bicep", "calf", "chest", "forearm", "height",
           "hip", "leg-length", "shoulder-breadth", "shoulder-to-crotch", "thigh", "waist", "wrist"]

def features(df):
    X = df[[h_col, w_col]].copy()
    X["male"] = (df[g_col].str.lower() == "male").astype(int)
    return X

rows = []
for t in targets:
    m = LinearRegression().fit(features(tr), tr[t])
    err = (m.predict(features(te)) - te[t]).abs()
    rows.append((t, err.mean(), (err <= 1.5).mean() * 100))
res = pd.DataFrame(rows, columns=["measurement", "MAE_cm", "% within ±1.5 cm"]).round(2)
print(res.to_string(index=False))

## What you now have
- `data[...]`: one row per photo pair, with inputs and answers
- the silhouette folders `mask/` (front) and `mask_left/` (side)
- a per-person height + weight baseline: the score to beat

Next: `notebooks/02_colab_pipeline.ipynb` trains the photo model.